# Robotics Control and Learning-Guided Motion Planning

This notebook is the primary technical reference for a two-part 4-DOF robotics project: dynamic simulation and PID control in MATLAB, followed by learning-guided trajectory sampling and MPC-style control in Python. It preserves the engineering content and supplied simulation evidence while distinguishing research theory from the implementation that is present in the repository.

The intended reading order is **theory -> source code -> visual result -> interpretation**.

## 1. Evidence and provenance model

Technical claims in this notebook use the following evidence classes.

| Label | Meaning |
|---|---|
| **Verified in source** | Directly visible in the tracked MATLAB or Python implementation |
| **Reported theory** | Derivation or method described in the supplied engineering material or referenced paper |
| **Preserved result** | Figure or numerical value supplied with the project, not recomputed here |
| **Reconstructed diagnostic** | A new read-only calculation made from tracked parameters and explicitly labeled as diagnostic |
| **Supplementary explanation** | Standard engineering context added to connect equations and code |
| **Unresolved** | A fact that cannot be verified from the available source and artifacts |

This distinction is important because the preserved figures appear to have been generated from an earlier source revision in several places.

### 1.1 Coverage of the supplied technical material

| Source-material topic | Notebook location | Coverage decision |
|---|---|---|
| Project purpose and two-part structure | Sections 2, 4, 5, 6, and 7 | Preserved and reorganized |
| Robot schematic, parameters, and generalized coordinates | Section 3 | Preserved, cross-checked, and extended with code-specific kinematics |
| Lagrangian energy terms and reported equations of motion | Section 4 | Preserved in compact form, including the identified prismatic-inertia inconsistency |
| Open-loop MATLAB simulation and figures | Section 4 | Source excerpt, supplied figures, and interpretation preserved |
| PID theory, tuning, MATLAB implementation, and figures | Section 5 | Preserved and extended with the corresponding discrete-time law |
| CVAE paper theory, offline/online stages, architecture, and hybrid sampling | Section 6 | Technically summarized and attributed; not presented as executed repository code |
| GMM trajectory model and MPC-style Python implementation | Section 7 | Verified from source and documented mathematically |
| Planning figures and reported experiments | Sections 7 and 8 | Supplied project figures preserved; unsupported benchmark claims are not repeated |
| Conclusions, advantages, limitations, and future work | Sections 10 and 11 | Consolidated with verified implementation limitations |

The supplied PDF bundle also embeds a full copy of the referenced research article and a presentation deck. Their project-relevant method, equations, architecture, claims, and citation are summarized here, but the external article and presentation are not reproduced page-for-page. Personal, institutional, grading, and submission metadata are intentionally excluded from this technical reference.

## 2. System overview

The repository contains two related but independent studies.

![Verified system architecture](../assets/images/system-architecture.png)

The Python code is an MPC-style sampled controller. It does not integrate the MATLAB dynamic model.

## 3. Robot geometry and generalized coordinates

The reported mechanism has three rotary joints and one prismatic joint, represented by

$$
q = [\theta_1,\theta_2,\theta_3,d_4]^T.
$$

![Reported kinematic structure](../assets/images/robot-arm-kinematic-structure.jpg)

The figure is a preserved project schematic. The MATLAB and Python implementations use different frame conventions, so the image should be treated as conceptual rather than as an exact frame-by-frame specification of both programs.

### 3.1 Code-specific forward kinematics

The Python planner does not construct Denavit-Hartenberg transforms. It directly evaluates joint positions. Defining $\phi_{12}=\theta_1+\theta_2$ and $\phi_{123}=\theta_1+\theta_2+\theta_3$, the implemented expressions can be written compactly as

$$
p_2(q)=\begin{bmatrix}L_2\cos\theta_1 & L_2\sin\theta_1 & L_1\end{bmatrix}^T,
$$

$$
p_3(q)=\begin{bmatrix}L_2\cos\theta_1+L_3\cos\phi_{12} & L_2\sin\theta_1+L_3\sin\phi_{12} & L_1+L_3\sin\theta_2\end{bmatrix}^T,
$$

$$
p_4(q)=p_3(q)+L_4\begin{bmatrix}\cos\phi_{123} & \sin\phi_{123} & \sin(\theta_2+\theta_3)\end{bmatrix}^T.
$$

The prismatic direction used by the source is normalized explicitly:

$$
\widetilde a(q)=\begin{bmatrix}\cos\phi_{123} & \sin\phi_{123} & \sin(\theta_2+\theta_3)\end{bmatrix}^T,\qquad a(q)=\frac{\widetilde a(q)}{\|\widetilde a(q)\|}.
$$

Thus the implemented end-effector map is

$$
p_{ee}(q)=p_4(q)+d_4a(q),\qquad J_{ee}(q)=\frac{\partial p_{ee}(q)}{\partial q}.
$$

The Jacobian is useful for differential kinematics and inverse-kinematics analysis, but the tracked controller does not evaluate it. The stored length $L_5$ is also not used by this forward-kinematics function. These equations document the code exactly; they are not asserted to be the unique physical frame convention of the mechanism.

### 3.2 Parameter cross-check

| Quantity | MATLAB source | Python source | Interpretation |
|---|---:|---:|---|
| Link lengths | 187.5, 200, 50, 200, 112.5 | 0.1125, 0.2, 0.05, 0.2, 0.1875 | MATLAB uses millimeter-scale numbers; Python uses meter-scale numbers, with a different ordering |
| Masses | 350, 341, 100, 400, 600 | 0.6, 0.4, 0.1, 0.341, 0.35 | Stored in both parts but unused by the Python planner |
| Stored inertias | 500, 700, 400, 500, 700 | 0.0007, 0.0005, 0.0004, 0.0005 for $I_2$ through $I_5$ | Values and scaling differ; physical equivalence cannot be established from the supplied units |
| Open-loop inputs | $[2,2,2,5]$ | Not applicable | First three entries are treated as rotary inputs and the fourth as prismatic input; source units are not encoded |
| Rotary limits | Not enforced | $[-\pi,\pi]$, then $[-\pi/2,\pi/2]$ | Verified in Python |
| Prismatic limit | Desired value 40 | 0 to 0.04 | Millimeters in MATLAB intent, meters in Python |
| Gravity | 9.81 | 9.81 | Stored in Python but not used by planning |

**Unresolved:** the historical table labels mass moments of inertia with units of $\mathrm{mm}^4$. A mass moment of inertia requires mass-length-squared units, such as $\mathrm{kg\,m^2}$ or $\mathrm{g\,mm^2}$. The original numerical values are preserved, but their physical units cannot be verified.

## 4. Part I - Lagrangian dynamics

### 4.1 Theory

The reported derivation begins with the Lagrangian

$$L(q,\dot q)=T(q,\dot q)-U(q),$$

where a rigid-body kinetic-energy contribution has the form

$$T_i=\frac{1}{2}m_i v_i^2+\frac{1}{2}I_i\dot\theta_i^2,$$

and the gravitational potential energy is

$$U=\sum_{i=1}^{5}m_i g h_i(q).$$

For the supplied simplified derivation, let $C_i=\cos\theta_i$ and $S_i=\sin\theta_i$. Its reported kinetic energy is

$$
T_{rep}=\frac{1}{2}I_2\dot\theta_1^2+\frac{1}{2}I_3\dot\theta_2^2+\frac{1}{2}I_4\dot\theta_3^2+\frac{1}{2}m_5\dot d_4^2.
$$

The corresponding center-of-mass heights are reported as

$$
h_1=\frac{L_1}{2},\qquad h_2=L_1+\frac{L_2C_1}{2},
$$

$$
h_3=L_1+L_2C_1+\frac{L_3}{2}(C_1C_2+S_1S_2),
$$

$$
h_4=L_1+L_2C_1+L_3(C_1C_2+S_1S_2)+\frac{L_4}{2}(C_1C_2C_3+S_1S_2C_3),
$$

$$
h_5=L_1+L_2C_1+L_3(C_1C_2+S_1S_2)+\left(L_4+\frac{d_4}{2}\right)(C_1C_2C_3+S_1S_2C_3).
$$

Euler-Lagrange equations then give

$$\frac{d}{dt}\left(\frac{\partial L}{\partial \dot q_i}\right)-\frac{\partial L}{\partial q_i}=\tau_i.$$

A compact form of the reported coordinate equations is therefore

$$
M_T\ddot q+\nabla_q U(q)=\tau,\qquad M_T=\mathrm{diag}(I_2,I_3,I_4,m_5).
$$

A conventional manipulator form is

$$M(q)\ddot q+C(q,\dot q)\dot q+G(q)=\tau.$$

The later matrix presentation instead uses $M_{rep}=\mathrm{diag}(I_2,I_3,I_4,I_5)$ and gives $I_5\ddot d_4=\tau_4-m_5g/2$. This conflicts with the $m_5\dot d_4^2/2$ kinetic term above. **Reported-theory limitation:** the supplied derivation also reduces $M$ to a constant diagonal matrix and groups configuration-dependent gravity expressions under a Coriolis label. It should therefore be presented as the project's simplified model, not as a complete rigid-body derivation.

### 4.2 Open-loop source implementation

The complete implementation is in [`src/matlab/part1_not_pid.m`](../src/matlab/part1_not_pid.m). The central update is: 

```matlab
inertiaMat = [inertia2 0 0 0;
              0 inertia3 0 0;
              0 0 inertia4 0;
              0 0 0 inertia5];

inputTorques = [2 2 2 5];
angularAccel = inertiaMat \ inputTorques' ...
             - inertiaMat \ (coriolisMat * anglesVel);

angle1Vel = angularAccel(1) * timeStep + angle1Vel;
angle1 = angularAccel(1) * timeStep^2 * 0.5 ...
       + angle1Vel * timeStep + angle1;
```

The variable named `coriolisMat` is a 4-by-1 expression built from gravity and configuration terms, while `anglesVel` is initialized once and never refreshed. Consequently, the source does not implement the conventional $C(q,\dot q)\dot q+G(q)$ decomposition shown above.

The source uses $\Delta t=0.1$ over a nominal 30 s interval. Its component-wise integration order is

$$
\dot q_{k+1}=\dot q_k+\ddot q_k\Delta t,
$$

$$
q_{k+1}=q_k+\dot q_{k+1}\Delta t+\frac{1}{2}\ddot q_k\Delta t^2.
$$

Because the updated velocity appears in the second equation, constant acceleration would produce $q_{k+1}=q_k+\dot q_k\Delta t+1.5\ddot q_k\Delta t^2$, rather than the standard constant-acceleration coefficient of $0.5$. This is the update actually encoded, not a recommended integrator.

### 4.3 Preserved open-loop results

![Open-loop joint positions](../assets/images/open-loop-position-response.png)

![Open-loop joint velocities](../assets/images/open-loop-velocity-response.png)

![Open-loop arm configuration](../assets/images/open-loop-arm-configuration.png)

### 4.4 Open-loop interpretation

The preserved curves show monotonically increasing coordinates and velocities under constant numerical inputs. This is qualitatively consistent with a continuously accelerated, undamped numerical model. However, the figures are evidence of a historical script run rather than a validated physical response: the parameter units are inconsistent, the gravity/velocity term does not match the conventional manipulator equation, and no raw arrays or execution log were supplied.

## 5. Part I - PID control

### 5.1 Theory

For reference $r(t)$ and measured output $y(t)$,

$$e(t)=r(t)-y(t),$$

and the continuous-time PID law is

$$u(t)=K_p e(t)+K_i\int_0^t e(\tau)d\tau+K_d\frac{de(t)}{dt}.$$

Its ideal transfer function is

$$G_c(s)=K_p+\frac{K_i}{s}+K_d s.$$

For a sampled controller, the intended coordinate-wise discretization is

$$
e_k=q_d-q_k,\qquad I_k=I_{k-1}+e_k\Delta t,\qquad D_k=\frac{e_k-e_{k-1}}{\Delta t},
$$

$$
\tau_k=K_pe_k+K_iI_k+K_dD_k.
$$

The reported target and manual tuning values are

$$
q_d=\begin{bmatrix}90^\circ & 90^\circ & 90^\circ & 40\;\mathrm{mm}\end{bmatrix}^T,
$$

with $K_p=1.5$ for every coordinate, $K_i=[30.595,14.2,19.68,29.85]$, and $K_d=[0.005,0.005,0.005,0.01]$. The proportional term responds to present error, the integral term removes persistent offset, and the derivative term adds rate-sensitive damping. The supplied discussion identifies manual tuning, Ziegler-Nichols tuning, and software-assisted optimization as possible tuning approaches; the stored gains are described as manually selected. Rotary and prismatic errors must be interpreted in their own units, because a single unscaled vector norm would mix degrees and millimeters.

### 5.2 PID source implementation

The complete implementation is in [`src/matlab/part1_pid.m`](../src/matlab/part1_pid.m).

```matlab
K = [1.5 1.5 1.5 1.5 ...
     30.595 14.2 19.68 29.85 ...
     0.005 0.005 0.005 0.01];

errorAngles = angles_d - angles;
integralError = integralError + errorAngles;
errorVel = (errorAngles - prevError) / timeStep;
torques = K(:,1:4).*errorAngles ...
        + K(:,5:8).*errorVel ...
        + K(:,9:12).*integralError;
```

**Verified source issues:**

- The documented integral gains are applied to the derivative term, and the documented derivative gains are applied to the integral term.
- `angles` is not updated after the joint coordinates change, so the error is not computed from the current simulated state.
- The integral accumulator is not multiplied by `timeStep`.
- `torques' * inputTorques` forms a 4-by-4 outer product rather than a four-element actuation vector.
- The prismatic position update uses `angularAccel(3)` instead of `angularAccel(4)`.
- The simulation stops when any coordinate exceeds its target, leaving trailing zeros in preallocated arrays.

In mathematical notation, the tracked accumulator and gain mapping are closer to

$$
I_k^{src}=I_{k-1}^{src}+e_k,\qquad \tau_k^{src}=K_pe_k+K_iD_k+K_dI_k^{src},
$$

which makes the difference from the intended discrete PID law explicit. This equation documents the verified implementation; it is not a corrected controller.

### 5.3 Preserved PID results

![PID joint positions](../assets/images/pid-position-response.png)

![PID joint velocities](../assets/images/pid-velocity-response.png)

![PID arm configuration](../assets/images/pid-arm-configuration.jpg)

The preserved numerical claim is

$$\theta_1=90.0025^\circ,\quad \theta_2=89.998^\circ,\quad \theta_3=90.0495^\circ,\quad d_4=40.3407\;\mathrm{mm}.$$

### 5.4 PID interpretation

The rising portions of the preserved curves are consistent with motion toward the requested coordinates. The subsequent drop to zero is not a physical return to the origin; it is explained by the early loop termination and untouched trailing elements in the preallocated arrays. The source narrative reports both 0.4 s and 10 s, while the plotted transition occurs at approximately 10 s. The terminal values are therefore retained as historical claims, not as independently reproduced controller performance.

If raw arrays become available, controller performance should be reported per coordinate using, for example,

$$
\mathrm{RMSE}_i=\sqrt{\frac{1}{N}\sum_{k=0}^{N-1}(q_{i,k}-q_{d,i})^2},\qquad e_{i,f}=|q_{i,N-1}-q_{d,i}|.
$$

Overshoot and settling time should likewise be computed separately for each rotary or prismatic coordinate. These definitions are supplementary evaluation criteria, not additional project results.

## 6. Part II - learning-based sampling theory

The referenced method addresses a central sampling-based motion-planning problem: uniform samples preserve broad coverage but spend many samples in regions that are unlikely to participate in a useful path. The reported alternative learns high-probability regions from successful demonstrations or simulations while retaining some uniform exploration.

### 6.1 Conditional generative model

A planning problem is represented by a condition such as

$$
y=\mathcal{E}(X_{init},X_{goal},X_{free}),
$$

where $X_{init}$ is the initial state, $X_{goal}$ is the goal region, $X_{free}$ describes free space, and $\mathcal{E}$ is a problem encoder. A Conditional Variational Autoencoder models

$$p(x\mid y)=\int p(x\mid z,y)p(z\mid y)\,dz,$$

where $x$ is a useful planning state, $y$ describes the problem, and $z$ is a latent variable. A conventional conditional encoder and decoder are

$$
q_\phi(z\mid x,y)=\mathcal{N}\!\left(\mu_\phi(x,y),\mathrm{diag}(\sigma_\phi^2(x,y))\right),
$$

$$
p_\theta(x\mid z,y)=\mathcal{N}\!\left(f_\theta(z,y),\sigma_x^2I\right).
$$

The reparameterization $z=\mu_\phi+\sigma_\phi\odot\epsilon$, with $\epsilon\sim\mathcal{N}(0,I)$, permits gradient-based training. Training maximizes an evidence lower bound (ELBO), conventionally written as

$$\mathcal{L}_{ELBO}=\mathbb{E}_{q_\phi(z\mid x,y)}[\log p_\theta(x\mid z,y)]-D_{KL}(q_\phi(z\mid x,y)\|p(z\mid y)).$$

A hybrid research sampler can be expressed as

$$
p_{hybrid}(x\mid y)=\lambda p_\theta(x\mid y)+(1-\lambda)p_{uniform}(x),\qquad 0\leq\lambda\leq1.
$$

The supplied method description uses $\lambda=0.5$ as a balance between learned concentration and broad state-space coverage.

### 6.2 Offline and online workflow

1. **Offline:** collect successful motion plans or demonstrations, construct $y$, and train the conditional encoder and decoder.
2. **Online:** encode a new planning problem, draw $z$ from the latent prior, decode task-aware states, mix learned and uniform samples, and pass them to a sampling-based motion planner.

The research contribution is the learned sampling distribution rather than replacement of the planner itself. The supplied article reports improved sample efficiency, solution cost, and success rate on several planning domains, including geometric, spacecraft, narrow-passage, workspace-learning, manipulator, and multirobot examples. Those paper benchmarks are external research evidence and are not experimental results of this repository.

### 6.3 Reported advantages and limitations

The method is attractive because it can focus computation on problem-relevant regions, condition on robot and environment information, and retain coverage through hybrid sampling. Its limitations include dependence on representative training data, nontrivial training cost, limited interpretability, sensitivity to unfamiliar environments, and the need to select the learned-sample ratio $\lambda$. Suggested future directions include more diverse data, adaptive $\lambda$, model compression, and online feedback.

**Theory-to-code boundary:** no encoder, decoder, latent network, ELBO optimization, occupancy-grid condition, or uniform sampling-based motion planner is implemented in this repository. The equations above explain the research inspiration, not the executed Python algorithm.

## 7. Part II - repository implementation

The full implementation is in [`src/python/part2_control.py`](../src/python/part2_control.py). It contains four main classes.

| Class | Responsibility |
|---|---|
| `HedgeTrimmingRobot` | Joint limits and forward kinematics |
| `Environment` | Three spherical obstacles and collision/proximity queries |
| `NormalizingFlowMPC` | Synthetic trajectory generation, GMM fitting, cost evaluation, and control selection |
| `MotionPlanningSimulation` | Three scenarios, state integration, figures, GIFs, and summary output |

The class name `NormalizingFlowMPC` is historical. The learned object is `sklearn.mixture.GaussianMixture`.

### 7.1 Synthetic training and GMM fitting

For each scenario, the program attempts to generate 200 trajectories using straight, via-point, random, and obstacle-avoidance heuristics. For a horizon of $N$ configurations, each feasible trajectory is flattened into

$$
\xi=\mathrm{vec}([q_0,q_1,\ldots,q_{N-1}])\in\mathbb{R}^{4N}.
$$

A full-covariance Gaussian mixture is then fitted:

$$
p(\xi)=\sum_{k=1}^{K}\pi_k\mathcal{N}(\xi\mid\mu_k,\Sigma_k),\qquad \pi_k\geq0,\qquad \sum_{k=1}^{K}\pi_k=1.
$$

The library estimates the mixture parameters by maximizing the training log-likelihood

$$
\max_{\{\pi_k,\mu_k,\Sigma_k\}}\sum_{n=1}^{N_{train}}\log p(\xi^{(n)}),\qquad K=\min\!\left(3,\left\lfloor\frac{N_{train}}{10}\right\rfloor\right).
$$

```python
training_data = np.array(training_trajectories)
self.learned_distribution = GaussianMixture(
    n_components=min(self.n_components, len(training_trajectories)//10),
    covariance_type='full',
    random_state=42
)
self.learned_distribution.fit(training_data)
```

This learns a distribution over complete discretized trajectories for one start-goal pair. At control time, nominally 25 complete trajectories are sampled from the GMM and 25 are produced by straight, via-point, or random heuristics. Thus the tracked 50-50 split is **learned plus heuristic**, not the learned-plus-uniform SBMP mixture described by the referenced method. Sampled trajectories are reshaped to $N\times4$, after which their first and last configurations are overwritten with the current state and goal. The model is not environment-conditioned and does not generalize across planning problems.

### 7.2 MPC-style objective and action selection

For candidate trajectory $q_0,\ldots,q_{N-1}$, define

$$
e_{q,t}=q_t-q_g,\qquad \Delta q_t=q_t-q_{t-1},\qquad w_t=1+5\frac{t}{N}.
$$

For spherical obstacles with centers $c_j$ and radii $r_j$, the end-effector clearance used in the soft penalty is

$$
d_t=\max\!\left(0,\min_j(\|p_{ee}(q_t)-c_j\|-r_j)\right).
$$

The obstacle term is piecewise quadratic:

$$
J_{obs,t}=\begin{cases}Q_{obs}(0.15-d_t)^2,&d_t<0.15,\\0,&d_t\geq0.15.\end{cases}
$$

The implemented objective can then be summarized as

$$
J=\sum_{t=0}^{N-1} \left[w_t e_{q,t}^TQ_{pos}e_{q,t}+50w_t\|p_{ee}(q_t)-p_{ee}(q_g)\|^2+\Delta q_t^TR\Delta q_t+J_{obs,t}+0.1\left\|\frac{\Delta q_t}{\Delta t_c}\right\|^2\right]+100\|q_{N-1}-q_g\|^2.
$$

Here $Q_{pos}=\mathrm{diag}(10,10,10,100)$, $R=\mathrm{diag}(0.1,0.1,0.1,1)$, $Q_{obs}=100$, and the controller uses $\Delta t_c=0.1$ s. The stored matrix `Q_vel` does not enter the cost. Candidates violating a joint limit or the source's point-collision predicate receive infinite cost. Selection is therefore

$$
\xi^*=\arg\min_{\xi\in\mathcal{C}}J(\xi),\qquad J(\xi)=\infty\;\text{if }\xi\text{ is infeasible}.
$$

The controller applies the first difference from the lowest-cost trajectory:

```python
control = (best_trajectory[1] - best_trajectory[0]) / self.dt
control *= 2.0
return np.clip(control, -1.0, 1.0)
```

Equivalently, the returned command is

$$
u_k=\mathrm{clip}\!\left(2\frac{q_1^*-q_0^*}{\Delta t_c},-1,1\right).
$$

The simulated state is then advanced and projected onto the joint limits as

$$
q_{k+1}=\Pi_{\mathcal Q}(q_k+u_k\Delta t_s),\qquad \Delta t_s=0.05\;\mathrm{s}.
$$

The controller horizon uses $\Delta t_c=0.1$ s while state execution uses $\Delta t_s=0.05$ s. There is no rigid-body transition, acceleration state, torque model, or numerical optimizer inside this control loop. `scipy.optimize.minimize` is used only by an approximate inverse-kinematics helper.

### 7.3 Collision-checking implementation

For reported joint position $p_i(q)$ and spherical obstacle $(c_j,r_j)$, define the point clearance

$$
\delta_{ij}(q)=\|p_i(q)-c_j\|-(r_j+m),\qquad m=0.02\;\mathrm{m}.
$$

The tracked collision predicate is $\min_{i,j}\delta_{ij}(q)\leq0$, implemented as

```python
for pos in positions:
    for obs in self.obstacles:
        if np.linalg.norm(pos - obs['center']) <= obs['radius'] + 0.02:
            return True
```

The source checks discrete joint positions. For comparison, the closest point on link segment $[a,b]$ to obstacle center $c$ is obtained from

$$
\alpha^*=\mathrm{clip}\!\left(\frac{(c-a)^T(b-a)}{\|b-a\|^2},0,1\right),\qquad p^*=a+\alpha^*(b-a),
$$

with segment clearance

$$
\delta_{seg}=\|p^*-c\|-(r+m).
$$

A segment collision occurs when $\delta_{seg}\leq0$. The tracked planner does not evaluate this expression; it is the supplementary geometry used to interpret the reconstructed diagnostic below.

### 7.4 Preserved planning results

#### Point-to-point

![Point-to-point task space](../assets/images/planning-task-space-point-to-point.png)

![Point-to-point configuration space](../assets/images/planning-configuration-space-point-to-point.png)

#### Complex maneuvering

![Complex task space](../assets/images/planning-task-space-complex-maneuvering.png)

![Complex configuration space](../assets/images/planning-configuration-space-complex-maneuvering.png)

### 7.5 Planning-result interpretation

The point-to-point figures show a smooth end-effector curve and convergence toward the displayed joint targets, which is qualitatively consistent with the goal-tracking and smoothness terms in the implemented cost. The complex trajectory contains non-monotonic corrections, especially in the prismatic coordinate, which is consistent with repeated candidate selection in the presence of obstacle penalties. These are preserved historical outputs, not newly reproduced benchmarks. Source screenshots associated with the figures differ from the tracked scenario definitions, so the exact run configuration remains unresolved.

### 7.6 Reconstructed collision diagnostic - result

A read-only diagnostic sampled 101 configurations along each direct start-goal interpolation and compared the source's endpoint checks with zero-radius link-segment checks using the same 0.02 m margin.

| Scenario | Endpoint-collision samples | Segment-collision samples | Missed by source check |
|---|---:|---:|---:|
| Point-to-Point | 0 | 0 | 0 |
| Obstacle Avoidance | 0 | 39 | 39 |
| Complex Maneuvering | 22 | 31 | 9 |

### 7.7 Reconstructed collision diagnostic - interpretation

The diagnostic shows that point-only collision checking can miss link-obstacle intersections, most clearly in the obstacle-avoidance scenario. These values are reconstructed diagnostics rather than preserved project results. They establish a limitation of the collision test, but they do not determine the collision status of every sampled candidate or every historical plotted trajectory.

## 8. Results inventory

For future reproducible planning comparisons, the principal metrics should include terminal configuration error, terminal task-space error, and minimum clearance:

$$
e_{q,f}=\|q_{N-1}-q_g\|,\qquad e_{p,f}=\|p_{ee}(q_{N-1})-p_{ee}(q_g)\|,
$$

$$
d_{min}=\min_{t,i,j}\left(\|p_i(q_t)-c_j\|-r_j\right).
$$

Success rate and cost distributions should be reported across repeated seeded runs. These definitions are evaluation guidance only because the raw historical trajectories and repeated-run data are unavailable.

| Artifact | Status | What can be claimed |
|---|---|---|
| Open-loop position and velocity figures | Preserved result | A historical script produced the displayed curves |
| Open-loop arm configuration | Preserved result | A historical script produced the displayed stick figure |
| PID position and velocity figures | Preserved result | Historical controller output with known trailing-zero behavior |
| PID terminal values | Reported numerical result | Values were supplied, but not independently reproduced |
| Planning task-space figures | Preserved result | Historical trajectories were plotted with three spherical obstacles |
| Planning configuration-space figures | Preserved result | Historical coordinates approached the displayed targets |
| Collision comparison table | Reconstructed diagnostic | Point-only checks can miss link-obstacle intersections |
| CVAE performance claims | Research theory only | Not a result of the tracked implementation |

No raw time series, random seeds for the complete historical runs, dependency lockfile, hardware timing log, or classical-planner baseline was supplied.

## 9. Reproduction guide

### MATLAB

Open MATLAB, change to `src/matlab/`, and run:

```matlab
run('part1_not_pid.m')
run('part1_pid.m')
```

The scripts open figures but do not save result files. Because the historical MATLAB version is unknown and the verified source issues affect numerical behavior, a new run should be labeled as a reproduction attempt rather than assumed to match the preserved figures.

### Python

From the repository root:

```bash
python -m venv .venv
python -m pip install -r requirements.txt
python src/python/part2_control.py
```

Activate the virtual environment before the final two commands. The output directory is created relative to the current working directory. Historical package versions are unresolved, so `requirements.txt` lists direct dependencies without claiming an exact lock.

## 10. Engineering limitations and recommended next work

### Verified limitations

1. Normalize all MATLAB parameters to one physical unit system.
2. Re-derive or validate $M(q)$, $C(q,\dot q)$, and $G(q)$ before interpreting dynamic performance.
3. Update state vectors every integration step and correct the PID gain mapping.
4. Replace the PID outer-product actuation with an explicit four-element vector.
5. Correct the prismatic acceleration index and integration rule.
6. Add segment or capsule collision checking for every robot link.
7. Rename the Python planner to reflect its GMM implementation, or implement the stated CVAE method as a separate module.
8. Add deterministic tests for forward kinematics, limits, collision queries, trajectory costs, and output creation.
9. Preserve raw result arrays and a machine-readable run manifest for every regenerated figure.
10. Compare learned sampling against an explicit uniform baseline before making performance claims.

These changes should be handled as a separate engineering revision. Correcting the source will change numerical behavior and would invalidate direct comparison with the preserved historical figures unless both versions are retained and labeled.

## 11. Conclusions

The repository demonstrates two useful robotics workflows: a compact MATLAB study of simplified joint dynamics and PID control, and a Python study of trajectory-distribution learning and MPC-style action selection. Its strongest educational value comes from connecting equations, source structure, and visualization.

The central documentation conclusion is equally important: the referenced CVAE method, the historical figures, and the tracked implementation are related but not identical. Treating them as separate evidence layers preserves the technical character of the project without overstating reproducibility or algorithmic capability.

### Referenced research method

Ichter, B., Harrison, J., and Pavone, M., *Learning Sampling Distributions for Robot Motion Planning*. The paper motivates conditional learned sampling for sampling-based motion planning; the tracked repository uses a simplified GMM trajectory model instead.